In [ ]:
## Phase 3 — TIES Merge Variants and Evaluation

**IMPORTANT: Restart kernel before running this phase.**
vLLM and Unsloth cannot share a Python session.

Seven variants evaluated:
- SM-TIES  · SM-Avg  (same-manifold)
- CM-TIES  · CM-Avg  · CMAM  (cross-manifold)
- SC-TIES  (shift-corrected)
- OP-TIES  (orthogonal projection)

Benchmarks:
- MMLU      n=5,000  SE=0.006  threshold=0.013
- HellaSwag n=2,000  SE=0.009  threshold=0.018
- GSM8K     n=500    SE=0.022  threshold=0.044

In [ ]:
import os
import gc
import json
import math
import torch
import random
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from datasets import load_dataset
from tqdm import tqdm

random.seed(42)
torch.manual_seed(42)

# ── Model paths ───────────────────────────────────────────
STUDENT_INSTRUCT = "Qwen/Qwen2.5-7B-Instruct"
STUDENT_BASE     = "Qwen/Qwen2.5-7B"

LORA_B1 = "/kaggle/working/lora_B1"
LORA_B2 = "/kaggle/working/lora_B2"
LORA_C1 = "/kaggle/working/lora_C1"
LORA_C2 = "/kaggle/working/lora_C2"

# ── Evaluation config ─────────────────────────────────────
MMLU_N       = 5000
HELLASWAG_N  = 2000
GSM8K_N      = 500
EVAL_SEED    = 42

# ── TIES config ───────────────────────────────────────────
TIES_DENSITY = 0.70   # ρ = top 70% by magnitude retained

# ── Results path ──────────────────────────────────────────
RESULTS_PATH = "/kaggle/working/results.json"

# Load existing results if session crashed
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, "r") as f:
        results = json.load(f)
    print(f"Loaded existing results: {list(results.keys())}")
else:
    results = {}
    print("No existing results found. Starting fresh.")

print("Config loaded.")

In [ ]:
CHAT_TEMPLATE = (
    "<|im_start|>user\n{msg}<|im_end|>\n"
    "<|im_start|>assistant\n"
)

def format_prompt(msg):
    return CHAT_TEMPLATE.format(msg=msg)

print("Chat template ready.")
print(f"Example:\n{format_prompt('What is 2+2?')}")

In [ ]:
def extract_task_vector(base_model_name, lora_path, dtype=torch.bfloat16):
    """
    Load LoRA onto base model, merge, extract task vector.
    Returns dict of {param_name: delta_tensor} on CPU.
    Zero disk usage.
    """
    print(f"  Extracting task vector: {lora_path}")

    # Load base
    base = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype    = dtype,
        device_map     = "cuda",
        trust_remote_code = True,
    )

    # Load base weights for delta computation
    base_state = {
        k: v.clone().cpu()
        for k, v in base.state_dict().items()
    }

    # Load LoRA onto base
    model = PeftModel.from_pretrained(base, lora_path)
    model = model.merge_and_unload()

    # Compute task vector
    task_vector = {
        k: model.state_dict()[k].cpu() - base_state[k]
        for k in base_state.keys()
    }

    del model, base, base_state
    gc.collect()
    torch.cuda.empty_cache()

    print(f"  Task vector extracted. Keys: {len(task_vector)}")
    return task_vector

In [ ]:
def ties_merge(task_vectors, density=TIES_DENSITY):
    """
    TIES merging: trim → elect → merge.
    
    Args:
        task_vectors: list of dicts {param_name: tensor}
        density:      fraction of parameters to retain (0.70)
    
    Returns:
        merged task vector dict
    """
    merged = {}
    conflict_count = 0
    total_count    = 0

    all_keys = task_vectors[0].keys()

    for key in all_keys:
        stacked = torch.stack([tv[key].float() for tv in task_vectors])

        # ── Step 1: Trim ──────────────────────────────────
        for i in range(len(task_vectors)):
            vec       = stacked[i]
            threshold = torch.quantile(vec.abs(), 1 - density)
            stacked[i] = torch.where(
                vec.abs() >= threshold,
                vec,
                torch.zeros_like(vec)
            )

        # ── Step 2: Elect sign ────────────────────────────
        sign_sum      = stacked.sum(dim=0)
        elected_sign  = torch.sign(sign_sum)
        elected_sign  = torch.where(
            elected_sign == 0,
            torch.ones_like(elected_sign),
            elected_sign
        )

        # ── Count conflict ────────────────────────────────
        signs = torch.sign(stacked)
        signs = torch.where(signs == 0, elected_sign.unsqueeze(0).expand_as(signs), signs)
        disagreement = (signs != elected_sign.unsqueeze(0)).float().mean().item()
        conflict_count += disagreement
        total_count    += 1

        # ── Step 3: Merge agreers ─────────────────────────
        agrees = (torch.sign(stacked) == elected_sign.unsqueeze(0)) | (stacked == 0)
        masked = torch.where(agrees, stacked, torch.zeros_like(stacked))
        counts = agrees.float().sum(dim=0).clamp(min=1)
        merged[key] = (masked.sum(dim=0) / counts).to(torch.bfloat16)

    sign_conflict = conflict_count / total_count if total_count > 0 else 0.0
    print(f"  TIES complete. Sign conflict: {sign_conflict:.4f}")
    return merged, sign_conflict


def average_merge(task_vectors):
    """Simple averaging baseline."""
    merged = {}
    all_keys = task_vectors[0].keys()
    for key in all_keys:
        stacked    = torch.stack([tv[key].float() for tv in task_vectors])
        merged[key] = stacked.mean(dim=0).to(torch.bfloat16)
    print("  Average merge complete.")
    return merged

In [ ]:
def build_merged_model(reference_model_name, merged_vector, dtype=torch.bfloat16):
    """
    Apply merged task vector to reference model.
    Returns model in VRAM ready for evaluation.
    """
    print(f"  Building merged model from {reference_model_name}")

    model = AutoModelForCausalLM.from_pretrained(
        reference_model_name,
        torch_dtype    = dtype,
        device_map     = "cuda",
        trust_remote_code = True,
    )

    state = model.state_dict()
    for key in merged_vector:
        if key in state:
            state[key] = (
                state[key].cpu().float() +
                merged_vector[key].float()
            ).to(dtype)

    model.load_state_dict(state)
    print("  Merged model ready in VRAM.")
    return model

In [ ]:
def evaluate_mmlu(model, tokenizer, n=MMLU_N, seed=EVAL_SEED):
    """Evaluate MMLU via first-token log-probability."""
    dataset = load_dataset(
        "cais/mmlu", "all",
        split="test"
    ).shuffle(seed=seed).select(range(n))

    choices    = ["A", "B", "C", "D"]
    choice_ids = [
        tokenizer.encode(c, add_special_tokens=False)[0]
        for c in choices
    ]

    correct = 0
    for item in tqdm(dataset, desc="MMLU"):
        prompt = format_prompt(
            f"{item['question']}\n"
            f"A. {item['choices'][0]}\n"
            f"B. {item['choices'][1]}\n"
            f"C. {item['choices'][2]}\n"
            f"D. {item['choices'][3]}\n"
            f"Answer:"
        )
        inputs = tokenizer(
            prompt,
            return_tensors = "pt"
        ).to("cuda")

        with torch.no_grad():
            logits = model(**inputs).logits[0, -1]

        pred = choices[torch.tensor(
            [logits[cid].item() for cid in choice_ids]
        ).argmax().item()]

        if pred == choices[item["answer"]]:
            correct += 1

    acc = correct / n
    print(f"  MMLU: {acc:.4f} ({correct}/{n})")
    return acc


def evaluate_hellaswag(model, tokenizer, n=HELLASWAG_N, seed=EVAL_SEED):
    """Evaluate HellaSwag via completion log-probability."""
    dataset = load_dataset(
        "Rowan/hellaswag",
        split="validation"
    ).shuffle(seed=seed).select(range(n))

    correct = 0
    for item in tqdm(dataset, desc="HellaSwag"):
        ctx     = item["ctx"]
        endings = item["endings"]
        label   = int(item["label"])

        scores = []
        for ending in endings:
            prompt = format_prompt(ctx + " " + ending)
            inputs = tokenizer(
                prompt,
                return_tensors = "pt"
            ).to("cuda")

            with torch.no_grad():
                out    = model(**inputs, labels=inputs["input_ids"])
                scores.append(-out.loss.item())

        if scores.index(max(scores)) == label:
            correct += 1

    acc = correct / n
    print(f"  HellaSwag: {acc:.4f} ({correct}/{n})")
    return acc


def evaluate_gsm8k(model, tokenizer, n=GSM8K_N, seed=EVAL_SEED):
    """Evaluate GSM8K via chain-of-thought generation."""
    dataset = load_dataset(
        "gsm8k", "main",
        split="test"
    ).shuffle(seed=seed).select(range(n))

    correct = 0
    for item in tqdm(dataset, desc="GSM8K"):
        prompt = format_prompt(item["question"])
        inputs = tokenizer(
            prompt,
            return_tensors = "pt"
        ).to("cuda")

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens = 256,
                do_sample      = False,
                temperature    = 1.0,
                pad_token_id   = tokenizer.eos_token_id,
            )

        generated = tokenizer.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        # Parse answer
        answer = None
        if "[number]" in generated:
            try:
                answer = generated.split("[number]")[-1].strip().split()[0]
            except:
                pass
        if answer is None:
            # Fallback: last number in output
            import re
            nums = re.findall(r'\d+\.?\d*', generated)
            answer = nums[-1] if nums else None

        # Get ground truth
        gt = item["answer"].split("####")[-1].strip()

        if answer and answer.replace(",","") == gt.replace(",",""):
            correct += 1

    acc = correct / n
    print(f"  GSM8K: {acc:.4f} ({correct}/{n})")
    return acc


def run_evaluation(model, tokenizer, variant_name):
    """Run all three benchmarks and save results."""
    print(f"\nEvaluating: {variant_name}")

    mmlu      = evaluate_mmlu(model, tokenizer)
    hellaswag = evaluate_hellaswag(model, tokenizer)
    gsm8k     = evaluate_gsm8k(model, tokenizer)

    results[variant_name] = {
        "MMLU":      mmlu,
        "HellaSwag": hellaswag,
        "GSM8K":     gsm8k,
    }

    # Save immediately after each variant
    with open(RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)

    print(f"\n{variant_name} results saved.")
    print(f"  MMLU:      {mmlu:.4f}")
    print(f"  HellaSwag: {hellaswag:.4f}")
    print(f"  GSM8K:     {gsm8k:.4f}")

    return mmlu, hellaswag, gsm8k

In [ ]:
if "Baseline" not in results:
    print("Evaluating baseline: Qwen2.5-7B-Instruct")

    tokenizer = AutoTokenizer.from_pretrained(
        STUDENT_INSTRUCT,
        trust_remote_code=True
    )

    baseline_model = AutoModelForCausalLM.from_pretrained(
        STUDENT_INSTRUCT,
        torch_dtype    = torch.bfloat16,
        device_map     = "cuda",
        trust_remote_code = True,
    )

    run_evaluation(baseline_model, tokenizer, "Baseline")

    del baseline_model
    gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"Baseline already evaluated: {results['Baseline']}")

In [ ]:
print("Extracting all task vectors...")
print("All stored in CPU RAM. No disk writes.")

# Same-manifold task vectors (reference: θ_inst)
print("\nSame-manifold task vectors:")
tau_B1 = extract_task_vector(STUDENT_INSTRUCT, LORA_B1)
tau_B2 = extract_task_vector(STUDENT_INSTRUCT, LORA_B2)

# Cross-manifold task vectors (reference: θ_base)
print("\nCross-manifold task vectors:")
tau_C1 = extract_task_vector(STUDENT_BASE, LORA_C1)
tau_C2 = extract_task_vector(STUDENT_BASE, LORA_C2)

# Anchor vector: τ_anchor = θ_inst − θ_base
print("\nComputing τ_anchor = θ_inst − θ_base...")
inst_state = AutoModelForCausalLM.from_pretrained(
    STUDENT_INSTRUCT,
    torch_dtype    = torch.bfloat16,
    device_map     = "cpu",
    trust_remote_code = True,
).state_dict()

base_state = AutoModelForCausalLM.from_pretrained(
    STUDENT_BASE,
    torch_dtype    = torch.bfloat16,
    device_map     = "cpu",
    trust_remote_code = True,
).state_dict()

tau_anchor = {
    k: inst_state[k].float() - base_state[k].float()
    for k in inst_state.keys()
}

del inst_state, base_state
gc.collect()

print("All task vectors ready.")
print(f"τ_B1, τ_B2: same-manifold")
print(f"τ_C1, τ_C2: cross-manifold")
print(f"τ_anchor:   instruct shift")

In [ ]:
if "SM-TIES" not in results:
    print("\n" + "="*60)
    print("SM-TIES — Same-Manifold TIES")
    print("Reference: θ_inst | Vectors: τ_B1, τ_B2")
    print("="*60)

    merged_tv, conflict = ties_merge([tau_B1, tau_B2])
    results["SM-TIES"] = results.get("SM-TIES", {})
    results["SM-TIES"]["sign_conflict"] = conflict

    model = build_merged_model(STUDENT_INSTRUCT, merged_tv)
    run_evaluation(model, tokenizer, "SM-TIES")

    del model, merged_tv
    gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"SM-TIES already evaluated: {results['SM-TIES']}")

In [ ]:
if "SM-Avg" not in results:
    print("\n" + "="*60)
    print("SM-Avg — Same-Manifold Averaging")
    print("Reference: θ_inst | Vectors: τ_B1, τ_B2")
    print("="*60)

    merged_tv = average_merge([tau_B1, tau_B2])

    model = build_merged_model(STUDENT_INSTRUCT, merged_tv)
    run_evaluation(model, tokenizer, "SM-Avg")

    del model, merged_tv
    gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"SM-Avg already evaluated: {results['SM-Avg']}")

In [ ]:
if "CM-TIES" not in results:
    print("\n" + "="*60)
    print("CM-TIES — Cross-Manifold TIES")
    print("Reference: θ_base | Vectors: τ_C1, τ_C2")
    print("Expected: sign conflict = 0.000")
    print("="*60)

    merged_tv, conflict = ties_merge([tau_C1, tau_C2])
    results["CM-TIES"] = results.get("CM-TIES", {})
    results["CM-TIES"]["sign_conflict"] = conflict

    model = build_merged_model(STUDENT_BASE, merged_tv)
    run_evaluation(model, tokenizer, "CM-TIES")

    del model, merged_tv
    gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"CM-TIES already evaluated: {results['CM-TIES']}")

In [ ]:
if "CM-Avg" not in results:
    print("\n" + "="*60)
    print("CM-Avg — Cross-Manifold Averaging")
    print("Reference: θ_base | Vectors: τ_C1, τ_C2")
    print("Expected: identical to CM-TIES")
    print("="*60)

    merged_tv = average_merge([tau_C1, tau_C2])

    model = build_merged_model(STUDENT_BASE, merged_tv)
    run_evaluation(model, tokenizer, "CM-Avg")

    del model, merged_tv
    gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"CM-Avg already evaluated: {results['CM-Avg']}")

In [ ]:
if "CMAM" not in results:
    print("\n" + "="*60)
    print("CMAM — Cross-Manifold Anchored Merging")
    print("Reference: θ_base | Vectors: τ_C1, τ_C2, τ_anchor")
    print("="*60)

    merged_tv, conflict = ties_merge([tau_C1, tau_C2, tau_anchor])
    results["CMAM"] = results.get("CMAM", {})
    results["CMAM"]["sign_conflict"] = conflict

    model = build_merged_model(STUDENT_BASE, merged_tv)
    run_evaluation(model, tokenizer, "CMAM")

    del model, merged_tv
    gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"CMAM already evaluated: {results['CMAM']}")

In [ ]:
if "SC-TIES" not in results:
    print("\n" + "="*60)
    print("SC-TIES — Shift-Corrected TIES")
    print("Step 1: subtract τ_anchor from CM vectors")
    print("Step 2: TIES merge residuals")
    print("Step 3: add τ_anchor back")
    print("="*60)

    # Step 1: subtract anchor
    print("  Subtracting τ_anchor from τ_C1 and τ_C2...")
    tau_C1_corrected = {
        k: tau_C1[k].float() - tau_anchor[k].float()
        for k in tau_C1.keys()
    }
    tau_C2_corrected = {
        k: tau_C2[k].float() - tau_anchor[k].float()
        for k in tau_C2.keys()
    }

    # Step 2: TIES on residuals
    merged_residual, conflict = ties_merge(
        [tau_C1_corrected, tau_C2_corrected]
    )
    results["SC-TIES"] = results.get("SC-TIES", {})
    results["SC-TIES"]["sign_conflict"] = conflict
    print(f"  Sign conflict after correction: {conflict:.4f}")

    # Step 3: add anchor back
    print("  Restoring τ_anchor...")
    merged_tv = {
        k: merged_residual[k].float() + tau_anchor[k].float()
        for k in merged_residual.keys()
    }

    # Build from θ_base + full merged vector
    model = build_merged_model(STUDENT_BASE, merged_tv)
    run_evaluation(model, tokenizer, "SC-TIES")

    del model, merged_tv, merged_residual
    del tau_C1_corrected, tau_C2_corrected
    gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"SC-TIES already evaluated: {results['SC-TIES']}")

In [ ]:
if "OP-TIES" not in results:
    print("\n" + "="*60)
    print("OP-TIES — Orthogonal Projection TIES")
    print("Projects task vectors onto orthogonal complement")
    print("of pretrained weight dominant subspace (k=16)")
    print("Requires float32 for SVD stability")
    print("="*60)

    K = 16  # top-k singular vectors to project out

    # Load base weights in float32 for SVD
    print("  Loading θ_inst in float32 for SVD...")
    ref_model = AutoModelForCausalLM.from_pretrained(
        STUDENT_INSTRUCT,
        torch_dtype       = torch.float32,
        device_map        = "cpu",
        trust_remote_code = True,
    )
    ref_state = ref_model.state_dict()
    del ref_model
    gc.collect()

    def orthogonal_project(delta, W0, k=K):
        """
        Project delta onto orthogonal complement of top-k
        singular subspace of W0.
        ΔW_projected = P_L · ΔW · P_R
        P_L = I - U_k U_k^T
        P_R = I - V_k V_k^T
        """
        if delta.dim() != 2:
            return delta

        W0_f = W0.float()
        d_f  = delta.float()

        try:
            U, S, Vh = torch.linalg.svd(W0_f, full_matrices=False)
            U_k  = U[:, :k]
            V_k  = Vh[:k, :].T

            P_L  = torch.eye(U_k.shape[0]) - U_k @ U_k.T
            P_R  = torch.eye(V_k.shape[0]) - V_k @ V_k.T

            return (P_L @ d_f @ P_R).to(torch.bfloat16)
        except:
            return delta.to(torch.bfloat16)

    # Project τ_B1 and τ_B2
    print("  Projecting τ_B1...")
    tau_B1_proj = {}
    for key in tau_B1.keys():
        if key in ref_state:
            tau_B1_proj[key] = orthogonal_project(
                tau_B1[key], ref_state[key]
            )
        else:
            tau_B1_proj[key] = tau_B1[key]

    print("  Projecting τ_B2...")
    tau_B2_proj = {}
    for key in tau_B2.keys():
        if key in ref_state:
            tau_B2_proj[key] = orthogonal_project(
                tau_B2[key], ref_state[key]
            )
        else:
            tau_B2_proj[key] = tau_B2[key]

    del ref_state
    gc.collect()

    # TIES on projected vectors
    merged_tv, conflict = ties_merge([tau_B1_proj, tau_B2_proj])
    results["OP-TIES"] = results.get("OP-TIES", {})
    results["OP-TIES"]["sign_conflict"] = conflict
    print(f"  Sign conflict after projection: {conflict:.4f}")

    model = build_merged_model(STUDENT_INSTRUCT, merged_tv)
    run_evaluation(model, tokenizer, "OP-TIES")

    del model, merged_tv, tau_B1_proj, tau_B2_proj
    gc.collect()
    torch.cuda.empty_cache()
else:
    print(f"OP-TIES already evaluated: {results['OP-TIES']}")

In [ ]:
print("\n" + "="*70)
print("KD-TIES COMPLETE RESULTS")
print("="*70)

baseline = results.get("Baseline", {})
b_mmlu   = baseline.get("MMLU", 0)
b_hs     = baseline.get("HellaSwag", 0)
b_gsm    = baseline.get("GSM8K", 0)

# Significance thresholds
SE_MMLU = 0.013
SE_HS   = 0.018
SE_GSM  = 0.044

print(f"\n{'Method':<12} {'MMLU':>8} {'HellaSwag':>12} {'GSM8K':>8} {'Conflict':>10}")
print("-" * 58)

for method in ["Baseline","SM-TIES","SM-Avg","CM-TIES",
               "CMAM","CM-Avg","SC-TIES","OP-TIES"]:
    if method not in results:
        continue

    r       = results[method]
    mmlu    = r.get("MMLU", 0)
    hs      = r.get("HellaSwag", 0)
    gsm     = r.get("GSM8K", 0)
    conf    = r.get("sign_conflict", None)

    mmlu_sig = "★" if abs(mmlu - b_mmlu) > SE_MMLU and method != "Baseline" else ""
    hs_sig   = "★" if abs(hs - b_hs)     > SE_HS   and method != "Baseline" else ""
    gsm_sig  = "★" if abs(gsm - b_gsm)   > SE_GSM  and method != "Baseline" else ""
    conf_str = f"{conf:.3f}" if conf is not None else "—"

    print(
        f"{method:<12} "
        f"{mmlu:.4f}{mmlu_sig:>2} "
        f"{hs:.4f}{hs_sig:>2}     "
        f"{gsm:.4f}{gsm_sig:>2} "
        f"{conf_str:>10}"
    )

print("\n★ = statistically significant vs baseline (diff > 2×SE)")
print(f"Thresholds: MMLU={SE_MMLU} · HellaSwag={SE_HS} · GSM8K={SE_GSM}")
print(f"\nResults saved to: {RESULTS_PATH}")